In [1]:
import ee
import folium
import geopandas as gd
import matplotlib.pyplot as plt
import numpy as np
import rasterio

print("\n-------ALL LIBRARIES LOADED SUCCESSFULLY--------")
print(f"rasterio version: {rasterio.__version__}")
print(f"geopandas version: {gd.__version__}")
print(f"folium version: {folium.__version__}")
print(f"google earth version: {ee.__version__}")


-------ALL LIBRARIES LOADED SUCCESSFULLY--------
rasterio version: 1.5.1
geopandas version: 1.1.4
folium version: 0.20.0
google earth version: 1.7.40


In [2]:
try:
    ee.Initialize(project="ee-undergrad-thesis-509519")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="ee-undergrad-thesis-509519")

In [3]:
import ee
import numpy as np
import matplotlib.pyplot as plt

ee.Initialize(project="ee-undergrad-thesis-509519")

WEST, SOUTH, EAST, NORTH = 37.48, 47.05, 37.68, 47.15
roi = ee.Geometry.BBox(WEST, SOUTH, EAST, NORTH)
TARGET_BANDS = ['B2','B3','B4','B8','B11','B12']
band_idx = {name: idx for idx, name in enumerate(TARGET_BANDS)}

image = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2024-01-01', '2024-06-01')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .select(TARGET_BANDS)
    .sort('system:time_start', False)
    .first()
    .reproject(crs='EPSG:4326', scale=10)
)

pixel_data = image.sampleRectangle(region = roi, defaultValue=0)
band_arrays = [np.array(pixel_data.get(b).getInfo()) for b in TARGET_BANDS]
arr_3d = np.stack(band_arrays, axis=-1)

arr_float = np.clip(arr_3d / 10000.0,0,1) # whats normalize surface reflectance

b2_blue  = arr_float[:, :, band_idx['B2']]
b3_green = arr_float[:, :, band_idx['B3']]
b4_red   = arr_float[:, :, band_idx['B4']]
b8_nir   = arr_float[:, :, band_idx['B8']]
b11_swir = arr_float[:, :, band_idx['B11']]

eps = 1e-10
ndbi = np.clip((b11_swir - b8_nir) / (b11_swir + b8_nir + eps), -1.0, 1.0)
ndvi = np.clip((b8_nir - b4_red) / (b8_nir + b4_red + eps), -1.0, 1.0)

feature_tensor = np.concatenate([
    arr_float,
    np.expand_dims(ndbi, axis=-1),
    np.expand_dims(ndvi, axis=-1) # does this mean its added on the back
], axis=-1)
print(f"validated feature tensor shape: {feature_tensor.shape}")

rgb_composite = np.clip(np.dstack([b4_red, b3_green, b2_blue]) * 2.5,0,1)#whats the 2.5 for
fig, axes = plt.subplots(2,2,figsize=(14,12)) # makes 2 x 2 contrast grid

im1 = axes[0,1].imshow(b8_nir, cmap = 'viridis')
axes[0,1].set_title("NIR (b8) - reflectance response", fontsize = 12, fontweight = 'bold')
axes[0, 1].axis('off')
plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)

im2 = axes[1, 0].imshow(b11_swir, cmap='magma')# Structural Concrete
axes[1, 0].set_title("SWIR-1 (B11) - Concrete/Urban Response", fontsize=12, fontweight='bold')
axes[1, 0].axis('off')
plt.colorbar(im2, ax=axes[1, 0], fraction=0.046, pad=0.04)

im3 = axes[1, 1].imshow(ndbi, cmap='coolwarm', vmin=-0.5, vmax=0.5) # NDBI Heatmap
axes[1, 1].set_title("Derived NDBI Index [(B11-B8)/(B11+B8)]", fontsize=12, fontweight='bold')

axes[1, 1].axis('off')
plt.colorbar(im3, ax=axes[1, 1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

EEException: Image.sampleRectangle: Too many pixels in sample; must be <= 262144. Got 2480878.